# 01 · Forward Modeling: From Fault Slip to Surface Motion

When a fault slips at depth, the surrounding crust deforms and the ground
surface moves. A *forward model* takes a slip distribution that we choose and
predicts the surface displacements it would produce. This is the foundation for
everything else in this tutorial series: inversion (later notebooks) is just
running this forward model "backwards" to learn the slip from the data.

## Learning objectives
- Create a discretized planar fault with `Fault.planar()`.
- Understand the linear forward model $\mathbf{d} = G\mathbf{m}$.
- Predict surface displacements two ways: by hand (`G @ m`) and with the
  one-line helper `fault.displacement()`.
- Visualize fault geometry, slip, and predicted displacements with
  `geodef.plot`.
- Compute the moment magnitude of a slip distribution.

## The forward and inverse problems

Two complementary questions sit at the heart of geodetic fault modeling:

- **Forward problem** — *given* model parameters (slip on the fault), predict
  the data (surface displacements). **This notebook.**
- **Inverse problem** — *given* data, estimate the model parameters that best
  explain them. Notebooks 03 onward.

Both are tied together by a single linear relationship:

$$ \mathbf{d} = G\,\mathbf{m} + \mathbf{e} $$

| symbol | meaning | shape |
| --- | --- | --- |
| $\mathbf{d}$ | observations (surface displacements) | $(N_\text{obs},)$ |
| $\mathbf{m}$ | model parameters (slip on each patch) | $(2N,)$ |
| $G$ | Green's function matrix (the physics) | $(N_\text{obs},\, 2N)$ |
| $\mathbf{e}$ | measurement error | $(N_\text{obs},)$ |

The matrix $G$ encodes the elastic physics: column $j$ is the surface
displacement produced by *one unit of slip* on patch $j$. `geodef` builds it
from the Okada (1985) analytical solution for a rectangular dislocation in an
elastic half-space. Notebook 02 opens up $G$ in detail; here we treat it as the
engine behind the forward model.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

# reload geodef automatically if you edit the source while experimenting
%load_ext autoreload
%autoreload 2

## 1. Build a discretized fault

We model a subduction-zone megathrust: a shallow-dipping rectangular fault,
180 km long and 90 km wide, with its center 25 km deep. We split it into a grid
of `n_length × n_width = 12 × 6 = 72` patches so that slip can vary across the
fault surface.

`Fault.planar()` takes the centroid location, orientation (strike, dip), size,
and the patch counts. **All distances are in meters; all angles in degrees.**

> **Coordinate conventions.** Geographic inputs are latitude, longitude, and
> depth (positive down, meters). Strike is degrees clockwise from north; dip is
> degrees down from horizontal. Internally `geodef` works in a local
> East/North/Up Cartesian frame and converts at the boundaries.

In [2]:
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)

nL, nW = fault.grid_shape
print(f"Engine:      {fault.engine}")
print(f"Patches:     {fault.n_patches}  ({nL} along-strike x {nW} down-dip)")
print(f"Patch area:  {fault.areas[0] / 1e6:.0f} km^2 each")
print(f"Depth range: {fault.centers[:, 2].min() / 1e3:.1f} - "
      f"{fault.centers[:, 2].max() / 1e3:.1f} km")

Engine:      okada
Patches:     72  (12 along-strike x 6 down-dip)
Patch area:  225 km^2 each
Depth range: 15.3 - 34.7 km


The fault is a dipping plane below the surface. Let's look at it in 3-D, colored
by depth. `geodef.plot.fault3d` returns a Matplotlib 3-D axes, so you can keep
customizing it after the call.

In [3]:
ax=geodef.plot.fault3d(
    fault, color_by="depth", view=(32, -75),
    title="Megathrust geometry (72 patches)",
)
plt.show()

/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99020/3725105982.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Choose a slip distribution

Slip on each patch has two components in the fault plane:

- **strike-slip** — horizontal motion along strike,
- **dip-slip** — motion up or down the dip direction (thrust or normal faulting).

`geodef` stores the full slip model as a single **blocked** vector of length
$2N$ — all strike-slip values first, then all dip-slip values:

$$ \mathbf{m} = [\,\underbrace{s^\text{ss}_0, \dots, s^\text{ss}_{N-1}}_{\text{strike-slip}},\ \underbrace{s^\text{ds}_0, \dots, s^\text{ds}_{N-1}}_{\text{dip-slip}}\,] $$

A megathrust earthquake is almost pure thrust, so we place a smooth **dip-slip**
asperity near the center of the fault and leave strike-slip at zero. Patches are
numbered with the along-strike index varying fastest, so patch `k = j*nL + i`
sits at strike index `i` and dip index `j`.

In [4]:
# patch grid indices
i = np.arange(fault.n_patches) % nL      # along-strike index
j = np.arange(fault.n_patches) // nL     # down-dip index

# smooth Gaussian asperity of thrust (dip-slip), peaking near the center
i0, j0 = (nL - 1) / 2, nW / 2
slip_dip = 3.0 * np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_strike = np.zeros(fault.n_patches)

print(f"Peak dip-slip: {slip_dip.max():.2f} m")

Peak dip-slip: 2.92 m


The input slip distribution, shown in the fault's own along-strike /
along-dip frame with the up-dip edge at the top (the default for
`plot.slip`); a black line marks the up-dip edge:

In [5]:
geodef.plot.slip(
    fault, slip_dip, cmap="YlOrRd",
    colorbar_label="Dip-slip (m)", title="Input slip distribution",
)
plt.show()

/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99020/289437995.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Predict surface displacement

Now the forward model. We place a grid of synthetic GNSS stations at the surface
and ask: *how does the ground move?*

We'll do it **twice** — first spelling out the matrix algebra, then with the
one-line `geodef` helper — to show they are exactly the same computation.

> **Watch the argument order.** Data classes such as `GNSS` take **longitude
> first, then latitude** (`GNSS(lon, lat, ...)`), matching file column order.
> The fault methods take **latitude first** (`fault.displacement(lat, lon, ...)`).

In [6]:
# grid of surface stations (build the arrays in lon, lat order)
sta_lon, sta_lat = np.meshgrid(
    np.linspace(98.5, 101.5, 8),
    np.linspace(-3.6, -0.4, 8),
)
sta_lon, sta_lat = sta_lon.ravel(), sta_lat.ravel()
print(f"{sta_lat.size} stations")

64 stations


### 3a. The explicit calculation: $\mathbf{d} = G\mathbf{m}$

`fault.greens_matrix()` builds $G$ for these stations. Its shape is $(3M, 2N)$:
three displacement components (E, N, U) at each of the $M$ stations in the rows,
and the $2N$ blocked slip parameters in the columns. We form the blocked slip
vector $\mathbf{m}$, multiply, then unpack the interleaved
`[e, n, u, e, n, u, ...]` result.

In [7]:
G = fault.greens_matrix(sta_lat, sta_lon)        # (3M, 2N)
m = np.concatenate([slip_strike, slip_dip])      # blocked [ss | ds], length 2N

d = G @ m                                         # the forward model
pred_e, pred_n, pred_u = d[0::3], d[1::3], d[2::3]

print(f"G shape: {G.shape}  ->  (3 x {sta_lat.size} stations, "
      f"2 x {fault.n_patches} patches)")
print(f"m shape: {m.shape}")
print(f"Max horizontal displacement: {np.hypot(pred_e, pred_n).max():.3f} m")
print(f"Max vertical displacement:   {np.abs(pred_u).max():.3f} m")

G shape: (192, 144)  ->  (3 x 64 stations, 2 x 72 patches)
m shape: (144,)
Max horizontal displacement: 0.481 m
Max vertical displacement:   0.374 m


### 3b. The one-liner: `fault.displacement()`

`geodef` wraps exactly the steps above. `fault.displacement()` builds $G$, forms
the blocked slip vector, multiplies, and returns the unpacked components — so the
result is identical to what we just computed by hand.

In [8]:
ue, un, uz = fault.displacement(
    sta_lat, sta_lon,
    slip_strike=slip_strike, slip_dip=slip_dip,
)

print("Identical to G @ m:",
      np.allclose([ue, un, uz], [pred_e, pred_n, pred_u]))

Identical to G @ m:

 True


## 4. Visualize the predicted displacements

To plot vectors we wrap the predictions in a `GNSS` object — the same container
you would use for real data. We draw the fault footprint with `plot.map` for
spatial context, then overlay the displacement field with `plot.vectors`:
horizontal arrows plus the vertical component as colored dots (with a colorbar).
Everything is in the local East/North frame centered on the fault, and the arrows
are drawn on top of the dots so they stay visible even where the dots are large.

In [9]:
n_sta = sta_lat.size
gnss = geodef.GNSS(
    sta_lon, sta_lat,                  # lon, lat
    pred_e, pred_n, pred_u,            # E, N, U displacements
    se=np.full(n_sta, 0.002),          # nominal uncertainties
    sn=np.full(n_sta, 0.002),
    su=np.full(n_sta, 0.005),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
geodef.plot.slip(fault, slip_dip, ax=axes[0], cmap="YlOrRd",
                 coords="geographic",
                 colorbar_label="Dip-slip (m)", title="Slip (model)")

# fault footprint for context, then the displacement field on top
geodef.plot.map(fault, ax=axes[1])
scale = 35.0 / np.hypot(pred_e, pred_n).max()   # arrows ~35 km at peak
geodef.plot.vectors(gnss, fault, ax=axes[1], components="both", scale=scale,
                    legend=True, scale_arrow=0.2, scale_arrow_label="0.2 m",
                    title="Predicted surface displacement")
plt.show()

/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99020/3902762058.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The horizontal arrows point toward the trench-ward, up-dip edge and the vertical
field shows uplift above the slip patch flanked by subsidence — the classic
signature of a buried thrust.

## 5. How big an earthquake is this?

The seismic moment is $M_0 = \mu \sum_k s_k A_k$ (shear modulus $\times$ slip
$\times$ area, summed over patches), and the moment magnitude follows from
$M_w = \tfrac{2}{3}\log_{10} M_0 - 6.07$. `geodef` computes both (default shear
modulus $\mu = 30$ GPa).

In [10]:
Mw = fault.magnitude(slip_dip)
M0 = fault.moment(slip_dip)
print(f"Seismic moment M0: {M0:.2e} N m")
print(f"Moment magnitude:  Mw {Mw:.2f}")

Seismic moment M0: 2.83e+20 N m
Moment magnitude:  Mw 7.56


## Exercises
1. **Geometry.** Re-run with `dip=45` and then `dip=80`. How do the horizontal
   vs. vertical patterns change? Which dip gives the strongest vertical signal
   directly above the fault?
2. **Depth.** Increase the centroid `depth` to `40e3`. How does the peak surface
   displacement change, and how does the signal spread out?
3. **Mechanism.** Make the slip pure strike-slip (put the asperity in
   `slip_strike` instead of `slip_dip`). Describe the new vector pattern.
4. **Discretization.** Rebuild with `n_length=24, n_width=12`. Does the predicted
   displacement change much? (This foreshadows notebook 02.)

## Checkpoint questions
1. In $\mathbf{d} = G\mathbf{m}$, what does a single *column* of $G$ represent?
2. Why is the slip vector length $2N$ rather than $N$?
3. The rows of $G$ are ordered `[e, n, u, e, n, u, ...]`. Why does
   `pred_u = d[2::3]` extract the vertical component?

## Common mistakes
- **Swapping latitude and longitude.** `GNSS`/`InSAR`/`Vertical` take
  `(lon, lat, ...)`; `fault.displacement` takes `(lat, lon, ...)`.
- **Wrong units.** Lengths and depths are in *meters*. `depth=25` means 25 m,
  not 25 km — use `25e3`.
- **Mis-ordering the slip vector.** It is *blocked* `[all strike-slip | all
  dip-slip]`, not interleaved per patch.

## Summary
- A fault is discretized into patches; `Fault.planar()` builds the grid.
- The forward model is linear: $\mathbf{d} = G\mathbf{m}$.
- `fault.displacement()` is a convenience wrapper around `G @ m`.
- `geodef.plot` renders geometry, slip, and displacements in a local frame.

**Next:** notebook 02 opens up the Green's matrix $G$ — what each column means,
how it is assembled patch by patch, and how its structure controls what we can
later *invert* for.